In [ ]:
import pandas as pd
import numpy as np
import cv2
import joblib
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from skimage.metrics import structural_similarity as ssim

# Load dataset
df = pd.read_excel("extracted_features_200.xlsx")

# Convert histogram string data to numpy arrays
def parse_hist(hist_str):
    return np.array(eval(hist_str))  # Convert string to array

df["Security_Mark_Hist"] = df["Security_Mark_Hist"].apply(parse_hist)
df["Green_Strip_Hist"] = df["Green_Strip_Hist"].apply(parse_hist)
df["Serial_Number_Hist"] = df["Serial_Number_Hist"].apply(parse_hist)
df["Gandhiji_Hist"] = df["Gandhiji_Hist"].apply(parse_hist)

# Ensure the dataset has the 'image_path' column
if "image_path" not in df.columns:
    raise KeyError("Column 'image_path' is missing in dataset. Ensure image paths are saved.")

# Feature extraction
X = np.array(df[["Security_Mark_Hist", "Green_Strip_Hist", "Serial_Number_Hist", "Gandhiji_Hist"]].values.tolist())
X = X.reshape(X.shape[0], -1)  # Flatten histograms

y = df['Label']  # Target label: 0 (fake) or 1 (real)

# Train KNN model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)

# Save trained model
joblib.dump(knn, "knn_currency_model.pkl")

# Function to process a new note image and classify it
def classify_currency(image_path):
    knn = joblib.load("knn_currency_model.pkl")
    
    # Load the image and preprocess
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Error loading image: {image_path}")
    
    img = cv2.resize(img, (700, 300))
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    equalized = cv2.equalizeHist(blurred)
    processed_img = cv2.Canny(equalized, 150, 255)

    # Define ROIs
    rois = {
        "security_mark": [(0, 56), (32, 150)],
        "green_strip": [(380, 0), (410, 300)],
        "serial_number": [(445, 240), (630, 300)],
        "gandhiji": [(152, 65), (385, 300)]
    }

    # Extract histogram features for each ROI
    extracted_hist = []
    for name, ((x1, y1), (x2, y2)) in rois.items():
        roi = processed_img[y1:y2, x1:x2]
        hist = cv2.calcHist([roi], [0], None, [256], [0, 256]).flatten()
        extracted_hist.append(hist)
    
    extracted_hist = np.hstack(extracted_hist).reshape(1, -1)
    
    # Predict using KNN
    hist_pred = knn.predict(extracted_hist)[0]
    
    # Compute SSIM with dataset
    ssim_scores = []
    for idx, row in df.iterrows():
        dataset_img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
        if dataset_img is not None:
            dataset_img = cv2.resize(dataset_img, (700, 300))
            dataset_ssim = ssim(processed_img, dataset_img)
            ssim_scores.append(dataset_ssim)
    
    avg_ssim = np.mean(ssim_scores) if ssim_scores else 0  # Avoid errors if no valid SSIM scores

    # Weighted decision: 0.70 * histogram match + 0.30 * SSIM
    final_score = (0.70 * hist_pred) + (0.30 * avg_ssim)
    final_label = 1 if final_score > 0.7 else 0  # Threshold for classification
    
    return final_label

# Example Usage:
result = classify_currency("dataset/500/500_f3.jpg")
print("Currency is", "Real" if result == 1 else "Fake")